In [6]:
import geopandas as gpd

In [ ]:
url_geojson_2020 = "https://raw.githubusercontent.com/americanpanorama/mapping-inequality-census-crosswalk/main/MIv3Areas_2020TractCrosswalk.geojson"

try:
    holc_gdf_2020 = gpd.read_file(url_geojson_2020)
    print("Data ingestion OK:", len(holc_gdf_2020))

except Exception as e:
    print(f"Error loading: {e}")

In [7]:
print(holc_gdf_2020.head())

   area_id        city state  city_survey   cat grade label   res    com  \
0      244  Birmingham    AL         True  Best     A    A1  True  False   
1      244  Birmingham    AL         True  Best     A    A1  True  False   
2      244  Birmingham    AL         True  Best     A    A1  True  False   
3      244  Birmingham    AL         True  Best     A    A1  True  False   
4      244  Birmingham    AL         True  Best     A    A1  True  False   

     ind     fill         GISJOIN        GEOID     calc_area  pct_tract  \
0  False  #76a865  G0100730010801  01073010801  3.459131e+05    0.06848   
1  False  #76a865  G0100730010802  01073010802  2.155205e+06    0.38611   
2  False  #76a865  G0100730010803  01073010803  2.322751e+04    0.00273   
3  False  #76a865  G0100730010804  01073010804  1.392696e+06    0.19737   
4  False  #76a865  G0100730010806  01073010806  8.334346e+05    0.06635   

                                            geometry  
0  MULTIPOLYGON (((-86.7722 33.48494,

- `GEOID`: code FIPS of 11 digits of the modern censal sectors
- `grade`: calification of HOLC (A, B, C, D)
- `pct_tract`: percentage of the modern censal sector that is covered for this poligon

In [ ]:
# to cross (modern and old) the borders of the neighbourhoods (they do not match after all these years)
holc_filtered = holc_gdf_2020[["GEOID", "grade", "pct_tract", "city", "state"]].copy()

# clean anomaly values (a veces hay áreas sin grado comercial/industrial)
holc_filtered = holc_filtered[holc_filtered["grade"].isin(["A", "B", "C", "D"])]

# order by geoid and the ny coverage area
holc_ordered = holc_filtered.sort_values(
    by=["GEOID", "pct_tract"], ascending=[True, False]
)
# delete duplicates
holc_dominant = holc_ordered.drop_duplicates(subset="GEOID", keep="first").copy()

# transformar la letra en un score
risk_map = {"A": 0, "B": 1, "C": 2, "D": 3}
holc_dominant["redlining_score"] = holc_dominant["grade"].map(risk_map)

print("Final dimensions (grade GEOID):", holc_dominant.shape)
print(holc_dominant.head())

Final dimensions (grade GEOID): (17309, 6)
           GEOID grade  pct_tract        city state  redlining_score
123  01073000100     C    0.25833  Birmingham    AL                2
188  01073000300     D    0.83974  Birmingham    AL                3
220  01073000400     D    0.27927  Birmingham    AL                3
221  01073000500     D    0.63944  Birmingham    AL                3
270  01073000700     D    0.41226  Birmingham    AL                3
